# HaLRTC 黑白视频修复算法
高精度低秩张量补全 (High-accuracy Low-Rank Tensor Completion)

In [1]:
import numpy as np
from skimage.metrics import structural_similarity as ssim
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os
import time
import pandas as pd

# 绘图字体设置
# 中文优先使用宋体，英文和数字使用 Times New Roman
plt.rcParams['font.sans-serif'] = ['SimSun', 'Times New Roman']
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.unicode_minus'] = False

def plot_time_curve(time_values, metric_values, y_label, save_path):
    plt.figure(figsize=(7, 5))
    plt.plot(time_values, metric_values, linewidth=2)
    plt.xlabel('Time (s)', fontname='Times New Roman', fontsize=14)
    plt.ylabel(y_label, fontname='Times New Roman', fontsize=14)
    plt.xticks(fontname='Times New Roman', fontsize=12)
    plt.yticks(fontname='Times New Roman', fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()

def ten2mat(tensor, mode):
    """张量按模展开为矩阵"""
    return np.reshape(np.moveaxis(tensor, mode, 0), (tensor.shape[mode], -1), order='F')

def mat2ten(mat, tensor_size, mode):
    """矩阵折叠为张量"""
    index = list()
    index.append(mode)
    for i in range(tensor_size.shape[0]):
        if i != mode:
            index.append(i)
    return np.moveaxis(np.reshape(mat, list(tensor_size[index]), order='F'), 0, mode)

def svt(mat, lambda0):
    """奇异值阈值化 (SVT)"""
    u, s, v = np.linalg.svd(mat, full_matrices=False)
    vec = s - lambda0
    vec[np.where(vec < 0)] = 0
    return np.matmul(np.matmul(u, np.diag(vec)), v)

def read_yuv420_gray(video_path, height, width):
    """读取 QCIF 黑白视频。

    优先按照 YUV420 文件读取，只使用 Y 分量。
    如果文件是纯灰度 raw yuv，也会自动兼容。
    """
    y_size = width * height
    uv_size = y_size // 4
    yuv420_frame_size = y_size + 2 * uv_size
    file_size = os.path.getsize(video_path)

    if file_size % yuv420_frame_size == 0:
        frame_num = file_size // yuv420_frame_size
        video = np.zeros((height, width, frame_num), dtype=np.uint8)
        with open(video_path, 'rb') as f:
            for k in range(frame_num):
                y = np.frombuffer(f.read(y_size), dtype=np.uint8)
                if y.size != y_size:
                    raise ValueError(f'读取第 {k + 1} 帧失败: {video_path}')
                video[:, :, k] = y.reshape((height, width))
                f.seek(2 * uv_size, 1)
        return video, frame_num

    if file_size % y_size == 0:
        frame_num = file_size // y_size
        video = np.fromfile(video_path, dtype=np.uint8)
        video = video.reshape((frame_num, height, width))
        video = np.transpose(video, (1, 2, 0))
        return video, frame_num

    raise ValueError(f'视频大小与 QCIF 灰度或 YUV420 格式不匹配: {video_path}')

def write_yuv420_gray(video, save_path):
    """把修复后的灰度视频保存为 YUV420 文件，UV 分量写为 128。"""
    video = np.clip(np.round(video), 0, 255).astype(np.uint8)
    height, width, frame_num = video.shape
    uv_size = height * width // 4

    with open(save_path, 'wb') as f:
        for k in range(frame_num):
            f.write(video[:, :, k].tobytes())
            f.write(np.full(uv_size, 128, dtype=np.uint8).tobytes())
            f.write(np.full(uv_size, 128, dtype=np.uint8).tobytes())

def calculate_video_psnr(original, recovered, maxP=255.0):
    """按帧计算 PSNR 后取平均。"""
    original = original.astype(float)
    recovered = recovered.astype(float)
    mse_per_frame = np.mean((original - recovered) ** 2, axis=(0, 1))
    psnr_per_frame = np.full_like(mse_per_frame, np.inf, dtype=float)
    valid = mse_per_frame > 0
    psnr_per_frame[valid] = 10 * np.log10(maxP ** 2 / mse_per_frame[valid])

    finite_values = psnr_per_frame[np.isfinite(psnr_per_frame)]
    if finite_values.size == 0:
        return np.inf
    return float(np.mean(finite_values))

def calculate_video_ssim(original, recovered, maxP=255.0):
    """逐帧计算 SSIM 后取平均。"""
    original = np.clip(original, 0, 255).astype(float)
    recovered = np.clip(recovered, 0, 255).astype(float)
    frame_num = original.shape[2]
    values = []

    for k in range(frame_num):
        value = ssim(original[:, :, k], recovered[:, :, k], data_range=maxP)
        if np.isfinite(value):
            values.append(value)

    if len(values) == 0:
        return np.nan
    return float(np.mean(values))

def safe_float(value):
    try:
        return float(value)
    except Exception:
        return np.nan

def safe_save_summary(summary_rows, result_dir, filename='实验总结.xlsx'):
    """优先保存 Excel；如果 Excel 写入失败，则自动保存 CSV。"""
    os.makedirs(result_dir, exist_ok=True)
    summary_df = pd.DataFrame(summary_rows)
    summary_df = summary_df.replace([np.inf, -np.inf], np.nan)

    excel_path = os.path.join(result_dir, filename)
    csv_path = os.path.join(result_dir, filename.replace('.xlsx', '.csv'))

    try:
        summary_df.to_excel(excel_path, index=False, engine='openpyxl', na_rep='NaN', inf_rep='Inf')
        return excel_path
    except Exception as e:
        summary_df.to_csv(csv_path, index=False, encoding='utf-8-sig', na_rep='NaN')
        error_path = os.path.join(result_dir, 'excel写入失败说明.txt')
        with open(error_path, 'w', encoding='utf-8') as f:
            f.write('Excel 写入失败，已改为保存 CSV。\n')
            f.write(f'错误信息: {repr(e)}\n')
        print(f'Excel 写入失败，已保存 CSV: {csv_path}')
        return csv_path

def empty_history():
    return pd.DataFrame(columns=[
        'iteration',
        'elapsed_time_seconds',
        'MSE',
        'RMSE',
        'PSNR',
        'SSIM'
    ])

def enrich_history(history, dataset, method, variant, missing_rate, seed, parameter_settings, status):
    if history is None:
        history = empty_history()
    history = history.copy()
    history.insert(0, 'dataset', dataset)
    history.insert(1, 'method', method)
    history.insert(2, 'variant', variant)
    history.insert(3, 'missing_rate', missing_rate)
    history.insert(4, 'seed', seed)
    history['parameter_settings'] = parameter_settings
    history['convergence_status'] = status
    return history


In [2]:
def HaLRTC(I, Omega, alpha, rho, maxiter, epsilon=1e-4, verbose=True):
    """
    HaLRTC 算法主函数

    参数:
    I: 原始黑白视频张量，尺寸为 height × width × frames
    Omega: 二值掩码，1 表示观测，0 表示缺失
    alpha: 核范数正则化参数
    rho: ADMM 惩罚参数
    maxiter: 最大迭代次数
    epsilon: 收敛阈值
    verbose: 是否打印进度

    返回:
    tensor_recovery: 修复后的视频张量
    psnr_list: 每次迭代的 PSNR
    ssim_value: 最终平均 SSIM
    elapsed_time: 运行时间
    """
    Omega = Omega.astype(bool)
    dim0 = I.ndim
    dim1, dim2, dim3 = I.shape
    maxP = 255.0

    position = np.where(Omega == 1)
    sparse_tensor = I.astype(float) * Omega
    tensor_hat = sparse_tensor.copy()

    Z = np.zeros((dim1, dim2, dim3, dim0))
    T = np.zeros((dim1, dim2, dim3, dim0))

    last_tensor = tensor_hat.copy()
    train_norm = max(np.linalg.norm(sparse_tensor), 1e-12)
    psnr_list = []
    mse_list = []
    rmse_list = []
    elapsed_list = []

    start_time = time.time()

    for iter in range(maxiter):
        for k in range(dim0):
            Z[:, :, :, k] = mat2ten(
                svt(ten2mat(tensor_hat + T[:, :, :, k] / rho, k), alpha / rho),
                np.array([dim1, dim2, dim3]), k
            )

        tensor_hat = np.mean(Z - T / rho, axis=3)
        tensor_hat[position] = sparse_tensor[position]

        for k in range(dim0):
            T[:, :, :, k] = T[:, :, :, k] + rho * (tensor_hat - Z[:, :, :, k])

        tensor_recovery = np.maximum(0, tensor_hat)
        tensor_recovery = np.minimum(maxP, tensor_recovery)

        mse_value = float(np.mean((I.astype(float) - tensor_recovery.astype(float)) ** 2))
        rmse_value = float(np.sqrt(mse_value))
        current_psnr = calculate_video_psnr(I, tensor_recovery, maxP=maxP)
        elapsed_list.append(time.time() - start_time)
        mse_list.append(mse_value)
        rmse_list.append(rmse_value)
        psnr_list.append(current_psnr)

        if verbose:
            print(f"Epoch = {iter + 1}, PSNR = {current_psnr:.4f} dB")

        tol = np.linalg.norm(tensor_hat - last_tensor) / train_norm
        last_tensor = tensor_hat.copy()

        if tol < epsilon:
            if verbose:
                print(f"\n在第 {iter + 1} 次迭代时收敛")
            break

    psnr_list = np.asarray(psnr_list, dtype=float)
    history = pd.DataFrame({
        'iteration': np.arange(1, len(psnr_list) + 1),
        'elapsed_time_seconds': np.asarray(elapsed_list, dtype=float),
        'MSE': np.asarray(mse_list, dtype=float),
        'RMSE': np.asarray(rmse_list, dtype=float),
        'PSNR': psnr_list,
        'SSIM': np.nan
    })
    tensor_recovery = np.maximum(0, tensor_hat)
    tensor_recovery = np.minimum(maxP, tensor_recovery)

    ssim_value = calculate_video_ssim(I, tensor_recovery, maxP=maxP)
    elapsed_time = time.time() - start_time

    if verbose:
        print(f"\n{'=' * 50}")
        print(f"最终 PSNR: {psnr_list[-1]:.4f} dB")
        print(f"最终 SSIM: {ssim_value:.4f}")
        print(f"运行时间: {elapsed_time:.2f} 秒")
        print(f"{'=' * 50}")

    return tensor_recovery, psnr_list, ssim_value, elapsed_time, history


In [3]:
# 参数设置
seedr = 920
np.random.seed(seedr)

# 需要依次运行的视频文件
video_files = [
    {
        'video_name': 'news_qcif',
        'filename': 'news_qcif_gray.yuv'
    },
    {
        'video_name': 'akiyo_qcif',
        'filename': 'akiyo_qcif_gray.yuv'
    }
]

# 自动识别当前运行目录
base_dir = os.getcwd()
candidate_dirs = [
    base_dir,
    os.path.join(base_dir, '视频实验')
]

video_dir = None
for candidate_dir in candidate_dirs:
    processed_dir = os.path.join(candidate_dir, '处理后视频')
    if all(os.path.exists(os.path.join(processed_dir, item['filename'])) for item in video_files):
        video_dir = candidate_dir
        break

if video_dir is None:
    raise FileNotFoundError('没有找到同时包含 news_qcif_gray.yuv 和 akiyo_qcif_gray.yuv 的处理后视频目录')

# 视频格式设置
w = 176
h = 144
fps = 30
show_idx = [40, 200]

# 算法参数
missing_rates = [0.8, 0.9, 0.95]
alpha_list = [1]
rho = 0.01
maxiter = 1000
epsilon = 1e-4

# 存储结果
all_results = []

for video_item in video_files:
    video_name = video_item['video_name']
    video_path = os.path.join(video_dir, '处理后视频', video_item['filename'])
    I, frame_num = read_yuv420_gray(video_path, h, w)

    print(f"\n{'#' * 70}")
    print(f"开始处理视频: {video_item['filename']}")
    print(f"视频路径: {video_path}")
    print(f"尺寸: {h} x {w} x {frame_num}")
    print(f"{'#' * 70}\n")

    for missing_rate in missing_rates:
        np.random.seed(seedr)
        Omega = np.random.rand(h, w, frame_num) > missing_rate
        observed = (I.astype(float) * Omega).astype(np.uint8)

        result_dir = os.path.join(
            video_dir,
            'results',
            'HaLRTC',
            f'{video_name}_{int(missing_rate * 100)}'
        )
        os.makedirs(result_dir, exist_ok=True)

        print(f"\n{'=' * 60}")
        print('HaLRTC 黑白视频修复实验')
        print(f"{'=' * 60}")
        print(f"视频: {video_path}")
        print(f"尺寸: {h} x {w} x {frame_num}")
        print(f"帧率: {fps}")
        print(f"缺失率: {missing_rate * 100:.1f}%")
        print(f"观测像素: {np.sum(Omega)} / {I.size}")
        print(f"输出目录: {result_dir}")
        print(f"{'=' * 60}\n")

        best_psnr = -np.inf
        best_alpha = None
        best_result = None
        summary_rows = []

        for alpha in alpha_list:
            print(f"\n{'-' * 60}")
            print(f"测试参数: video={video_name}, missing_rate={missing_rate:.2f}, alpha={alpha}, rho={rho}")
            print(f"{'-' * 60}\n")

            status = 'ok'
            error_msg = ''
            parameter_settings = f'alpha={alpha}, rho={rho}, maxiter={maxiter}, epsilon={epsilon}'

            try:
                X_halrtc, psnr_halrtc, ssim_halrtc, elapsed_time, history_halrtc = HaLRTC(
                    I, Omega, alpha, rho, maxiter, epsilon
                )

                X_halrtc = np.clip(np.round(X_halrtc), 0, 255).astype(np.uint8)
                final_mse = float(np.mean((I.astype(float) - X_halrtc.astype(float)) ** 2))
                final_rmse = float(np.sqrt(final_mse))
                final_psnr = calculate_video_psnr(I, X_halrtc, maxP=255.0)
                final_ssim = calculate_video_ssim(I, X_halrtc, maxP=255.0)

                if not np.isfinite(final_psnr):
                    status = 'nan_metric'

            except Exception as e:
                status = 'error'
                error_msg = repr(e)
                print(f"参数 alpha={alpha} 运行失败，已跳过。错误: {error_msg}")

                X_halrtc = observed.copy()
                psnr_halrtc = np.array([], dtype=float)
                history_halrtc = empty_history()
                final_mse = np.nan
                final_rmse = np.nan
                final_psnr = np.nan
                final_ssim = np.nan
                elapsed_time = 0.0

            history_halrtc = enrich_history(
                history_halrtc,
                video_name,
                'HaLRTC',
                'default',
                missing_rate,
                seedr,
                parameter_settings,
                status
            )

            summary_rows.append({
                'Video': video_name,
                'MissingRate': missing_rate,
                'alpha': alpha,
                'rho': rho,
                'MSE': final_mse,
                'RMSE': final_rmse,
                'PSNR': final_psnr,
                'SSIM': final_ssim,
                'Time': elapsed_time,
                'Status': status,
                'Error': error_msg
            })

            print(f"最终 PSNR: {final_psnr:.4f} dB" if np.isfinite(final_psnr) else "最终 PSNR: NaN")
            print(f"最终 SSIM: {final_ssim:.4f}" if np.isfinite(final_ssim) else "最终 SSIM: NaN")
            print(f"运行时间: {elapsed_time:.2f} 秒")
            print(f"状态: {status}")

            if status == 'ok' and np.isfinite(final_psnr) and final_psnr > best_psnr:
                best_psnr = final_psnr
                best_alpha = alpha
                best_result = {
                    'original': I,
                    'observed': observed,
                    'recovered': X_halrtc,
                    'psnr_list': psnr_halrtc,
                    'history': history_halrtc,
                    'final_mse': final_mse,
                    'final_rmse': final_rmse,
                    'final_psnr': final_psnr,
                    'ssim': final_ssim,
                    'time': elapsed_time,
                    'status': status
                }

        if best_result is None:
            print('警告: 当前视频和缺失率下，所有参数都没有得到有限 PSNR。将保存观测视频作为占位结果，并继续后续实验。')
            best_alpha = None
            best_result = {
                'original': I,
                'observed': observed,
                'recovered': observed,
                'psnr_list': np.array([], dtype=float),
                'history': enrich_history(empty_history(), video_name, 'HaLRTC', 'default', missing_rate, seedr, 'all_failed', 'all_failed'),
                'final_mse': np.nan,
                'final_rmse': np.nan,
                'final_psnr': np.nan,
                'ssim': np.nan,
                'time': 0.0,
                'status': 'all_failed'
            }

        # 保存修复视频和原始结果
        write_yuv420_gray(best_result['recovered'], os.path.join(result_dir, f'{video_name}_recovered.yuv'))
        np.savez(
            os.path.join(result_dir, 'result.npz'),
            video_name=video_name,
            video_path=video_path,
            I=I,
            Omega=Omega,
            observed=observed,
            recovered=best_result['recovered'],
            psnr_list=best_result['psnr_list'],
            elapsed_time_seconds=np.asarray(best_result['history']['elapsed_time_seconds'], dtype=float),
            mse_list=np.asarray(best_result['history']['MSE'], dtype=float),
            rmse_list=np.asarray(best_result['history']['RMSE'], dtype=float),
            best_alpha=best_alpha,
            rho=rho
        )

        history_path = os.path.join(result_dir, 'best_iteration_history.csv')
        best_result['history'].to_csv(history_path, index=False, encoding='utf-8-sig', na_rep='NaN')

        # 保存指标
        metrics_path = os.path.join(result_dir, 'metrics.txt')
        with open(metrics_path, 'w', encoding='utf-8') as f:
            f.write(f'video_name={video_name}\n')
            f.write('method=HaLRTC\n')
            f.write('variant=default\n')
            f.write(f'video={video_path}\n')
            f.write(f'size={h} x {w} x {frame_num}\n')
            f.write(f'fps={fps}\n')
            f.write(f'missing_rate={missing_rate:.2f}\n')
            f.write(f'seed={seedr}\n')
            f.write(f'alpha={best_alpha}\n')
            f.write(f'rho={rho}\n')
            f.write(f'mse={safe_float(best_result["final_mse"]):.6f}\n' if np.isfinite(safe_float(best_result['final_mse'])) else 'mse=NaN\n')
            f.write(f'rmse={safe_float(best_result["final_rmse"]):.6f}\n' if np.isfinite(safe_float(best_result['final_rmse'])) else 'rmse=NaN\n')
            f.write(f'psnr={safe_float(best_result["final_psnr"]):.6f}\n' if np.isfinite(safe_float(best_result['final_psnr'])) else 'psnr=NaN\n')
            f.write(f'ssim={safe_float(best_result["ssim"]):.6f}\n' if np.isfinite(safe_float(best_result['ssim'])) else 'ssim=NaN\n')
            f.write(f'time={safe_float(best_result["time"]):.6f}\n' if np.isfinite(safe_float(best_result['time'])) else 'time=NaN\n')
            f.write(f'status={best_result["status"]}\n')
            f.write(f'history_file={history_path}\n')

        summary_path = safe_save_summary(summary_rows, result_dir)
        print(f"指标表已保存: {summary_path}")

        # 保存指定帧对比图
        show_idx_valid = [idx for idx in show_idx if idx <= frame_num]
        for idx in show_idx_valid:
            ori = I[:, :, idx - 1]
            obs = observed[:, :, idx - 1]
            rec = best_result['recovered'][:, :, idx - 1]

            fig, axes = plt.subplots(1, 3, figsize=(12, 4))

            axes[0].imshow(ori, cmap='gray')
            axes[0].set_title(f'原始视频帧 {idx}', fontsize=14)
            axes[0].axis('off')

            axes[1].imshow(obs, cmap='gray')
            axes[1].set_title(f'观测数据 ({missing_rate * 100:.0f}% 缺失)', fontsize=14)
            axes[1].axis('off')

            axes[2].imshow(rec, cmap='gray')
            axes[2].set_title(
                f'HaLRTC 修复\nPSNR={best_result["final_psnr"]:.2f}dB, SSIM={best_result["ssim"]:.4f}\n时间={best_result["time"]:.2f}s',
                fontsize=14
            )
            axes[2].axis('off')

            plt.tight_layout()
            plt.savefig(os.path.join(result_dir, f'frame_{idx:03d}_compare.png'), dpi=150, bbox_inches='tight')
            plt.close(fig)

        if not best_result['history'].empty:
            plot_time_curve(
                best_result['history']['elapsed_time_seconds'],
                best_result['history']['MSE'],
                'MSE',
                os.path.join(result_dir, 'mse_time_curve.png')
            )
            plot_time_curve(
                best_result['history']['elapsed_time_seconds'],
                best_result['history']['RMSE'],
                'RMSE',
                os.path.join(result_dir, 'rmse_time_curve.png')
            )

        print(f"\n视频 {video_name}，缺失率 {missing_rate * 100:.1f}% 的结果已保存")

        all_results.append({
            'video_name': video_name,
            'video_path': video_path,
            'missing_rate': missing_rate,
            'result_dir': result_dir,
            'best_alpha': best_alpha,
            'rho': rho,
            'best_result': best_result,
            'summary_rows': summary_rows
        })

print(f"\n{'=' * 60}")
print('所有视频和缺失率处理完成！')
print(f"{'=' * 60}")



######################################################################
开始处理视频: news_qcif_gray.yuv
视频路径: C:\Users\GIGA\Desktop\实验\视频实验\处理后视频\news_qcif_gray.yuv
尺寸: 144 x 176 x 300
######################################################################


HaLRTC 黑白视频修复实验
视频: C:\Users\GIGA\Desktop\实验\视频实验\处理后视频\news_qcif_gray.yuv
尺寸: 144 x 176 x 300
帧率: 30
缺失率: 80.0%
观测像素: 1520401 / 7603200
输出目录: C:\Users\GIGA\Desktop\实验\视频实验\results\HaLRTC\news_qcif_80


------------------------------------------------------------
测试参数: video=news_qcif, missing_rate=0.80, alpha=1, rho=0.01
------------------------------------------------------------



Epoch = 1, PSNR = 9.5877 dB


Epoch = 2, PSNR = 9.6053 dB


Epoch = 3, PSNR = 9.6230 dB


Epoch = 4, PSNR = 9.6407 dB


...

Epoch = 999, PSNR = 16.0977 dB


Epoch = 1000, PSNR = 16.1081 dB



最终 PSNR: 16.1081 dB
最终 SSIM: 0.4891
运行时间: 1973.77 秒


最终 PSNR: 16.1079 dB
最终 SSIM: 0.4888
运行时间: 1973.77 秒
状态: ok
指标表已保存: C:\Users\GIGA\Desktop\实验\视频实验\results\HaLRTC\akiyo_qcif_95\实验总结.xlsx



视频 akiyo_qcif，缺失率 95.0% 的结果已保存

所有视频和缺失率处理完成！


In [4]:
# 单独导出总 Excel 汇总
if 'all_results' not in globals():
    raise NameError('当前内核中没有 all_results，请先运行上一单元生成实验结果。')

total_summary = []

for item in all_results:
    data = item['best_result']
    total_summary.append({
        'Video': item['video_name'],
        'MissingRate': item['missing_rate'],
        'BestAlpha': item['best_alpha'],
        'rho': item['rho'],
        'MSE': data['final_mse'],
        'RMSE': data['final_rmse'],
        'PSNR': data['final_psnr'],
        'SSIM': data['ssim'],
        'Time': data['time'],
        'Status': data['status'],
        'ResultDir': item['result_dir']
    })

total_result_dir = os.path.join(video_dir, 'results', 'HaLRTC')
summary_path = safe_save_summary(total_summary, total_result_dir, filename='全部实验总结.xlsx')
print(f'总 Excel 已导出到: {summary_path}')


总 Excel 已导出到: C:\Users\GIGA\Desktop\实验\视频实验\results\HaLRTC\全部实验总结.xlsx
